In [9]:
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, r2_score
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import numpy as np
import seaborn as sns
from sklearn import preprocessing
from sklearn.svm import SVC
from sklearn.cluster import KMeans

xve = ''
yve = 'click'
fpv = 'ad_click_dataset.csv'
#imports data
origData = pd.read_csv(fpv)
df = origData
#displays first 5 rows
print(df.head())
#checks for missing values
print(df.isnull().any())

#rejected is not filled in so we are filling it in
df.fillna({'admission':"rejected"}, inplace=True)
#fills missing values
df.fillna(df.mean(numeric_only=True).round(1), inplace=True)
string_columns = df.select_dtypes(include=['object']).columns
df[string_columns] = df[string_columns].fillna(df[string_columns].mode().iloc[0])
#print(df.head())
#print(df.isnull().any())

#oneHotEncoding
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[categorical_columns])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns))
df_encoded = pd.concat([df, one_hot_df], axis=1)
df_encoded = df_encoded.drop(categorical_columns, axis=1)
print(df_encoded.head())

#getting columns in x
abc = df_encoded.columns.get_loc(yve)
test2 = []
test3 = []
for i in range(len(df.columns)):
    if(i != abc):
        test2.append(i)
limit = 5
for i in range(len(test2)):
    if(i<limit):
        test3.append(df_encoded.columns[test2[i]])


#sets x and y
X = df_encoded[test3]
y = df_encoded[yve]
X.to_numpy()
y.to_numpy()


     id full_name   age      gender device_type ad_position browsing_history  \
0   670   User670  22.0         NaN     Desktop         Top         Shopping   
1  3044  User3044   NaN        Male     Desktop         Top              NaN   
2  5912  User5912  41.0  Non-Binary         NaN        Side        Education   
3  5418  User5418  34.0        Male         NaN         NaN    Entertainment   
4  9452  User9452  39.0  Non-Binary         NaN         NaN     Social Media   

  time_of_day  click  
0   Afternoon      1  
1         NaN      1  
2       Night      1  
3     Evening      1  
4     Morning      0  
id                  False
full_name           False
age                  True
gender               True
device_type          True
ad_position          True
browsing_history     True
time_of_day          True
click               False
dtype: bool
     id   age  click  full_name_User10  full_name_User100  full_name_User1000  \
0   670  22.0      1               0.0                

array([1, 1, 1, ..., 0, 1, 0], dtype=int64)

In [10]:
#80/20 train test split with random seed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#first model
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
#printing scores/evaulation
print(f'Accuracy: {accuracy}')
print(f'F1: {f1}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print('Classification Report:')
print(classification_report(y_test, y_pred))

#second model
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
#printing scores/evaulation
print(f'Accuracy (Random Forest): {accuracy_rf}')
print(f'F1 (Random Forest): {f1_rf}')
print(f'Precision (Random Forest): {precision_rf}')
print(f'Recall (Random Forest): {recall_rf}')
print('Classification Report (Random Forest):')
print(classification_report(y_test, y_pred_rf))


Accuracy: 0.6475
F1: 0.7860394537177542
Precision: 0.6475
Recall: 1.0
Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       705
           1       0.65      1.00      0.79      1295

    accuracy                           0.65      2000
   macro avg       0.32      0.50      0.39      2000
weighted avg       0.42      0.65      0.51      2000



c:\Users\enson\OneDrive\Desktop\NewBeginnings\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\enson\OneDrive\Desktop\NewBeginnings\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\enson\OneDrive\Desktop\NewBeginnings\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

Accuracy (Random Forest): 0.9215
F1 (Random Forest): 0.9423853211009174
Precision (Random Forest): 0.8979020979020979
Recall (Random Forest): 0.9915057915057915
Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.98      0.79      0.88       705
           1       0.90      0.99      0.94      1295

    accuracy                           0.92      2000
   macro avg       0.94      0.89      0.91      2000
weighted avg       0.93      0.92      0.92      2000



In [11]:
#k fold implementation
